# Audit transcript terhadap OCR mentah
Pilih Runtime > Change runtime type > GPU. Upload transcript_photo_fresh_ocr_bundle.zip. Jalankan semua sel. Foto, transcript dan algoritma tidak diubah. Ini hanya OCR dan audit, bukan grid search atau tuning evaluasi.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, json, hashlib, os
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
root = Path('/content/lensalabel_transcript_audit')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    z.extractall(root)
os.chdir(root)
manifest = json.loads(Path('audit_bundle_manifest.json').read_text())
for name, digest in manifest['sha256'].items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest() == digest, name
print('Input verified:', len(manifest['sha256']))
del uploaded


In [ ]:
%pip install -q -r evaluation/requirements-colab.txt openpyxl
import torch
assert torch.cuda.is_available(), 'Aktifkan GPU sebelum OCR 160 foto.'
os.environ['EASYOCR_GPU'] = '1'
print(torch.cuda.get_device_name(0))


In [ ]:
import subprocess, sys, importlib.metadata, datetime
out = Path('evaluation/results/transcript_photo_audit')
out.mkdir(parents=True, exist_ok=True)
runtime = {'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'python': sys.version, 'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'easyocr': importlib.metadata.version('easyocr'), 'max_ocr_width': 2000, 'split_size': 160}
(out / 'colab_runtime.json').write_text(json.dumps(runtime, indent=2))
subprocess.run([sys.executable, '-u', 'evaluation/run_ocr_cache.py', '--split-csv', 'evaluation/splits/evaluation_set.csv', '--photos-dir', 'dataset/raw_photos', '--cache-dir', str(out / 'fresh_evaluation_ocr')], check=True)
subprocess.run([sys.executable, 'evaluation/audit_transcript_fresh.py'], check=True)
for name, digest in manifest['sha256'].items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest() == digest, name
print('Protected inputs remain unchanged')


In [ ]:
import shutil
result = shutil.make_archive('/content/transcript_photo_fresh_ocr_results', 'zip', root_dir='evaluation/results', base_dir='transcript_photo_audit')
files.download(result)
